# **Home Electricity Bill Forecasting with Time Series**

In [23]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from keras.layers import Dense, LSTM
import matplotlib.pyplot as plt
import tensorflow as tf
import joblib

### **Data Load and Assesement**

In [2]:
# load data
df = pd.read_csv('Consume.csv')
df.head()

,ID,Month,Days,Cost
0,1,July,30,99.642
1,2,August,31,93.694
2,3,September,30,96.668
3,4,October,31,101.103
4,5,November,30,104.115


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ID      19 non-null     int64  
 1   Month   19 non-null     object 
 2   Days    19 non-null     int64  
 3   Cost    19 non-null     float64
dtypes: float64(1), int64(2), object(1)
memory usage: 740.0+ bytes


In [4]:
# Drop ID
df_copy = df.copy()
df = df.drop(columns=['ID'])
df.head()

,Month,Days,Cost
0,July,30,99.642
1,August,31,93.694
2,September,30,96.668
3,October,31,101.103
4,November,30,104.115


In [7]:
# Statistic Descriptive
df.describe()

,Days,Cost
count,19.000000,19.000000
mean,30.473684,98.823000
std,0.772328,2.973225
min,28.000000,93.694000
25%,30.000000,96.784000
50%,31.000000,99.100000
75%,31.000000,100.801500
max,31.000000,104.115000


### **Data Pre-Processing**

In [10]:
# Encoding
encoder = OneHotEncoder(sparse_output=False)
month_enco = encoder.fit_transform(df[['Month']])

# Scaler
scaler = MinMaxScaler()
cost_scaled = scaler.fit_transform(df[['Cost']])

In [24]:
joblib.dump(encoder, 'encoder_ohe.pkl')
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

In [11]:
# Gabungkan fetaures (Month OHE + Days)
X = np.hstack((month_enco, df[['Days']].values))
y = cost_scaled

In [13]:
# Reshape X untuk LSTM : (samples, time_steps, features)
X = X.reshape((X.shape[0], 1, X.shape[1]))

In [14]:
X

array([[[ 0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,
         30.]],

       [[ 0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         31.]],

       [[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,
         30.]],

       [[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,
         31.]],

       [[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,
         30.]],

       [[ 0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         31.]],

       [[ 0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         31.]],

       [[ 0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         28.]],

       [[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,
         31.]],

       [[ 1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         30.]],

       [[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,
         31.]],

       [[ 0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0., 

### **Modelling**

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1], X.shape[2])),
    LSTM(64, activation='relu', return_sequences=False),
    Dense(32, activation='relu'),
    Dense(1) # Output prediksi biaya
])

model.compile(optimizer='adam', loss='mse')
model.fit(X, y, epochs=100, verbose=1)

# Simpan model
model.save('model_listrik.h5')

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - loss: 0.3286
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - loss: 0.2214
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - loss: 0.1437
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - loss: 0.0954
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - loss: 0.0735
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - loss: 0.0725
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - loss: 0.0845
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - loss: 0.1002
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - loss: 0.1122
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - loss: 0.1169
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step - loss: 0.1141
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - loss: 0.1059
Epoch 13/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - loss: 0.0950
Epoch 14/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - loss: 0.0840
Epoch 15/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - loss: 0.0749
Epoch 1

In [19]:
model.save('model_listrik.keras')

### **Inference**

In [22]:
# --- DI FILE SEPARATE / SAAT INFERENCE ---
loaded_model = tf.keras.models.load_model('model_listrik.keras')

def prediksi_bulan_ini(nama_bulan, jumlah_hari):
    # 1. Encode bulan (Harus pakai encoder yang sama saat training)
    bulan_test = pd.DataFrame({'Month': [nama_bulan]})
    bulan_ohe = encoder.transform(bulan_test) # encoder dari proses training
    
    # 2. Gabungkan dengan Days
    input_data = np.hstack((bulan_ohe, [[jumlah_hari]]))
    
    # 3. Reshape untuk LSTM
    input_data = input_data.reshape((1, 1, input_data.shape[1]))
    
    # 4. Predict
    prediksi_scaled = loaded_model.predict(input_data)
    
    # 5. Balikkan nilai ke Rupiah (Inverse Scale)
    hasil_asli = scaler.inverse_transform(prediksi_scaled)
    return hasil_asli[0][0]

# Contoh Penggunaan:
print(f"Prediksi biaya: {prediksi_bulan_ini('January', 31)}")

d:\python\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 9 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 511ms/step
Prediksi biaya: 99.4359359741211
